In [1]:
from confluent_kafka import Producer
import json
import time

# Configuration du Producer
conf = {
    "bootstrap.servers": "127.0.0.1:9092",  # Assure-toi d'utiliser 127.0.0.1
    "security.protocol": "PLAINTEXT",        # Assure-toi qu'il n'y a pas de protocole sécurisé
    "client.id": "python-producer"
}

producer = Producer(conf)

def delivery_report(err, msg):
    if err:
        print(f"⚠️ Erreur : {err}")
    else:
        print(f"✅ Envoyé à {msg.topic()} [{msg.partition()}]")

try:
    while True:
        data = {"temperature": 22, "city": "Paris"}
        producer.produce(
            topic="weather_data_demo",
            key="Paris",
            value=json.dumps(data),
            callback=delivery_report
        )
        producer.flush()  # Cette ligne permet de s'assurer que le message est envoyé
        print("📤 Message envoyé, pause de 5s...")
        time.sleep(5)
except KeyboardInterrupt:
    print("\n🛑 Arrêt du Producer.")


✅ Envoyé à weather_data_demo [0]
📤 Message envoyé, pause de 5s...
✅ Envoyé à weather_data_demo [0]
📤 Message envoyé, pause de 5s...

🛑 Arrêt du Producer.


In [2]:
from confluent_kafka import Consumer, KafkaError
import json

# Configuration du Consumer
conf = {
    'bootstrap.servers': '127.0.0.1:9092',
    'group.id': 'mon-groupe-consumer',
    'auto.offset.reset': 'earliest',  # Pour lire depuis le début du topic
}

# Création du Consumer
consumer = Consumer(conf)

# Abonnement au topic "weather_data_demo"
consumer.subscribe(['weather_data_demo'])

try:
    while True:
        # On attend un message pendant 1 seconde
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            # Gérer les erreurs
            if msg.error().code() == KafkaError._PARTITION_EOF:
                print(f"Fin de partition {msg.topic()} [{msg.partition()}]")
            else:
                print(f"Erreur: {msg.error()}")
        else:
            # Décodage et affichage du message
            value = msg.value().decode('utf-8')
            try:
                data = json.loads(value)
                print("Message reçu :", data)
            except json.JSONDecodeError:
                print("Message reçu (non-JSON):", value)
except KeyboardInterrupt:
    print("Arrêt du consumer")
finally:
    consumer.close()


Message reçu : {'temperature': 22, 'city': 'Paris'}
Message reçu : {'temperature': 22, 'city': 'Paris'}
Arrêt du consumer


%6|1740619306.884|FAIL|python-producer#producer-1| [thrd:127.0.0.1:9092/bootstrap]: 127.0.0.1:9092/1: Disconnected (after 615752ms in state UP)
%6|1740619306.998|FAIL|python-producer#producer-1| [thrd:127.0.0.1:9092/bootstrap]: 127.0.0.1:9092/1: Disconnected while requesting ApiVersion: might be caused by incorrect security.protocol configuration (connecting to a SSL listener?) or broker version is < 0.10 (see api.version.request) (after 4ms in state APIVERSION_QUERY)
%6|1740619307.285|FAIL|python-producer#producer-1| [thrd:127.0.0.1:9092/bootstrap]: 127.0.0.1:9092/1: Disconnected while requesting ApiVersion: might be caused by incorrect security.protocol configuration (connecting to a SSL listener?) or broker version is < 0.10 (see api.version.request) (after 4ms in state APIVERSION_QUERY, 1 identical error(s) suppressed)
%3|1740619312.555|FAIL|python-producer#producer-1| [thrd:127.0.0.1:9092/bootstrap]: 127.0.0.1:9092/1: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (afte